# 2026 호르무즈 해협(호르무즈 해협) 영어 해외언론 텍스트마이닝 수집

**기간:** 2026-01-01 ~ 2026-08-31  
**핵심 키워드:** `"Strait of Hormuz"`  
**수집 구조:** NewsCatcher → URL 중복 제거 → Diffbot Article → Diffbot Discussion → CSV/Excel  
**주의:** API 키는 코드에 직접 입력하지 않고 환경변수로 넣습니다.

> 이 노트북은 기존 `pygooglenews + Selenium` 수집부를 API 기반으로 교체하고,
> 기존 NLTK/텍스트마이닝 분석부와 연결하기 위한 수집용 노트북입니다.


## 0. 설치

Jupyter/VS Code에서 최초 1회만 실행합니다.


In [1]:
%pip install -q requests pandas openpyxl tqdm python-dotenv nltk scikit-learn matplotlib wordcloud

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. 라이브러리와 기본 설정

NewsCatcher API는 `https://v3-api.newscatcherapi.com/api/search`를 사용하고,
API 키는 `x-api-token` 헤더로 전달합니다.

Diffbot Article API는 원문 URL에서 제목, 본문, 날짜, 저자, 언론사명,
언론사 국가/지역, 언어 등을 추출할 수 있으며 댓글도 `discussion`으로 반환할 수 있습니다.


In [2]:
import os
import re
import json
import time
import hashlib
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

import requests
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()

NEWSCATCHER_API_KEY = os.getenv("NEWSCATCHER_API_KEY")
DIFFBOT_API_KEY = os.getenv("DIFFBOT_API_KEY")

if not NEWSCATCHER_API_KEY:
    raise ValueError(
        "NEWSCATCHER_API_KEY가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요."
    )

if not DIFFBOT_API_KEY:
    raise ValueError(
        "DIFFBOT_API_KEY가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요."
    )

# API 키 자체는 출력하지 않습니다.
print("API 키 로드 완료")
print(f"NewsCatcher API: {'설정됨' if NEWSCATCHER_API_KEY else '없음'}")
print(f"Diffbot API: {'설정됨' if DIFFBOT_API_KEY else '없음'}")

START_DATE = "2026-01-01"
END_DATE = "2026-08-31"

QUERIES = [
    '"Strait of Hormuz"',
    '"Hormuz Strait"',
]

NEWS_URL = "https://v3-api.newscatcherapi.com/api/search"
DIFFBOT_ARTICLE_URL = "https://api.diffbot.com/v3/article"
DIFFBOT_DISCUSSION_URL = "https://api.diffbot.com/v3/discussion"

DATA_DIR = Path("hormuz_2026_data")
DATA_DIR.mkdir(exist_ok=True)

RAW_NEWS_FILE = DATA_DIR / "01_newscatcher_raw.csv"
ARTICLES_FILE = DATA_DIR / "02_articles.csv"
COMMENTS_FILE = DATA_DIR / "03_comments.csv"
FAILED_FILE = DATA_DIR / "04_failed_urls.csv"


API 키 로드 완료
NewsCatcher API: 설정됨
Diffbot API: 설정됨


c:\Users\황태하\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. API 연결 테스트

실제 대량 수집 전에 **각 API 키가 정상인지 먼저 확인**합니다.


In [3]:
def test_newscatcher():
    headers = {
        "x-api-token": NEWSCATCHER_API_KEY,
        "Content-Type": "application/json",
    }
    payload = {
        "q": '"Strait of Hormuz"',
        "lang": "en",
        "from_": START_DATE,
        "to_": END_DATE,
        "page_size": 1,
    }
    r = requests.post(NEWS_URL, headers=headers, json=payload, timeout=60)
    print("NewsCatcher:", r.status_code)
    r.raise_for_status()
    data = r.json()
    print("total_hits:", data.get("total_hits"))
    return data

def test_diffbot(test_url="https://www.bbc.com/news"):
    params = {
        "token": DIFFBOT_API_KEY,
        "url": test_url,
        "discussion": "false",
    }
    r = requests.get(DIFFBOT_ARTICLE_URL, params=params, timeout=60)
    print("Diffbot:", r.status_code)
    r.raise_for_status()
    return r.json()

nc_test = test_newscatcher()


NewsCatcher: 200
total_hits: 10000


## 3. URL 정규화

검색어가 여러 개이면 같은 기사가 반복될 수 있으므로 URL을 정규화한 뒤 중복 제거합니다.


In [4]:
TRACKING_PARAMS = {
    "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
    "gclid", "fbclid", "mc_cid", "mc_eid"
}

def normalize_url(url):
    if not isinstance(url, str) or not url.strip():
        return None

    url = url.strip()
    parts = urlsplit(url)

    if parts.scheme not in {"http", "https"}:
        return None

    query_items = []
    for item in parts.query.split("&") if parts.query else []:
        if "=" in item:
            k, v = item.split("=", 1)
        else:
            k, v = item, ""
        if k.lower() not in TRACKING_PARAMS:
            query_items.append(f"{k}={v}")

    clean = urlunsplit((
        parts.scheme.lower(),
        parts.netloc.lower(),
        parts.path.rstrip("/") or "/",
        "&".join(query_items),
        ""
    ))
    return clean


## 4. NewsCatcher 기사 목록 수집

- 영어: `lang=en`
- 연구 기간: 2026-01-01 ~ 2026-08-31
- 페이지 크기: 최대 1000
- 검색어별 결과를 합친 뒤 URL 기준으로 중복 제거
- 검색 결과가 많으면 **날짜 구간을 자동으로 분할**합니다.

### 왜 날짜 분할이 필요한가?

NewsCatcher는 한 검색 조건에서 반환 가능한 기사 수에 상한이 있습니다.
따라서 2026-01-01 ~ 2026-08-31을 한 번에 검색하면 최신 기사부터 채워지고
과거 기사가 잘릴 수 있습니다.

이 코드는 먼저 전체 구간의 `total_hits`를 확인하고,
안전 기준을 초과하면 월/주/일 단위로 자동 분할하여 다시 검색합니다.


In [5]:
def _request_newscatcher(payload, max_retries=5):
    """NewsCatcher 요청 + 일시적 오류(429/5xx) 재시도."""
    headers = {
        "x-api-token": NEWSCATCHER_API_KEY,
        "Content-Type": "application/json",
    }

    for attempt in range(max_retries):
        try:
            r = requests.post(
                NEWS_URL,
                headers=headers,
                json=payload,
                timeout=90,
            )

            if r.status_code == 429:
                if attempt == max_retries - 1:
                    raise RuntimeError(
                        "NewsCatcher rate limit(429)입니다. "
                        "요금제 제한 또는 요청량을 확인하세요."
                    )
                wait = min(60, 2 ** attempt * 2)
                print(f"  429 발생 → {wait}초 후 재시도")
                time.sleep(wait)
                continue

            if 500 <= r.status_code < 600:
                if attempt == max_retries - 1:
                    r.raise_for_status()
                wait = min(60, 2 ** attempt * 2)
                print(f"  서버 오류({r.status_code}) → {wait}초 후 재시도")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r.json()

        except requests.RequestException:
            if attempt == max_retries - 1:
                raise
            wait = min(60, 2 ** attempt * 2)
            time.sleep(wait)

    raise RuntimeError("NewsCatcher 요청에 실패했습니다.")


def newscatcher_search(
    q,
    from_date=START_DATE,
    to_date=END_DATE,
    page=1,
    page_size=1000,
):
    payload = {
        "q": q,
        "lang": "en",
        "from_": from_date,
        "to_": to_date,
        "page": page,
        "page_size": page_size,
        "exclude_duplicates": True,
        "sort_by": "date",
    }
    return _request_newscatcher(payload)


def _date_range_days(from_date, to_date):
    start = pd.Timestamp(from_date)
    end = pd.Timestamp(to_date)
    return (end - start).days + 1


# 한 검색 구간에 너무 많은 기사가 있으면 더 작은 날짜 구간으로 분할합니다.
# 10,000건 제한에 정확히 걸리지 않도록 9,000건을 안전 기준으로 사용합니다.
NEWS_MAX_SAFE_HITS = 9000


def collect_newscatcher_chunk(
    q,
    from_date,
    to_date,
    depth=0,
):
    """한 날짜 구간을 검색하고, 결과가 많으면 재귀적으로 날짜를 쪼갭니다."""
    indent = "  " * depth

    first = newscatcher_search(
        q,
        from_date=from_date,
        to_date=to_date,
        page=1,
        page_size=1000,
    )

    total_hits = int(first.get("total_hits") or 0)
    total_pages = int(first.get("total_pages") or 1)

    print(
        f"{indent}[{q}] {from_date} ~ {to_date} | "
        f"hits={total_hits:,} | pages={total_pages}"
    )

    # 검색 결과가 안전 기준 이내이면 정상적으로 페이지를 모두 가져옵니다.
    if total_hits <= NEWS_MAX_SAFE_HITS:
        rows = first.get("articles", [])

        for page in range(2, total_pages + 1):
            data = newscatcher_search(
                q,
                from_date=from_date,
                to_date=to_date,
                page=page,
                page_size=1000,
            )
            rows.extend(data.get("articles", []))
            time.sleep(0.3)

        return rows

    # 하루까지 쪼갰는데도 너무 많다면 해당 API 검색 조건에서는
    # 더 이상 날짜로 분할할 수 없으므로 오류를 알립니다.
    if _date_range_days(from_date, to_date) <= 1:
        raise RuntimeError(
            f"하루 단위에서도 {total_hits:,}건이 검색됩니다: "
            f"{q} / {from_date} ~ {to_date}"
        )

    start = pd.Timestamp(from_date)
    end = pd.Timestamp(to_date)
    mid = start + (end - start) // 2

    left_from = start.strftime("%Y-%m-%d")
    left_to = mid.strftime("%Y-%m-%d")
    right_from = (mid + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    right_to = end.strftime("%Y-%m-%d")

    print(
        f"{indent}→ 결과가 많아 날짜를 분할합니다: "
        f"{left_from}~{left_to} / {right_from}~{right_to}"
    )

    left_rows = collect_newscatcher_chunk(
        q, left_from, left_to, depth + 1
    )
    right_rows = collect_newscatcher_chunk(
        q, right_from, right_to, depth + 1
    )

    return left_rows + right_rows


def collect_newscatcher(q, from_date=START_DATE, to_date=END_DATE):
    return collect_newscatcher_chunk(q, from_date, to_date)


# 검색어별로 전체 기간을 수집합니다.
all_raw = []

for q in QUERIES:
    try:
        rows = collect_newscatcher(q)
        print(f"[{q}] 최종 수집: {len(rows):,}건")
        all_raw.extend(rows)
    except Exception as e:
        print(f"검색 실패: {q} -> {e}")

raw_df = pd.json_normalize(all_raw)

print("검색어 통합 raw 기사 수:", len(raw_df))

# API 원본은 그대로 저장해 두고, 이후 단계에서 URL 중복을 제거합니다.
raw_df.to_csv(RAW_NEWS_FILE, index=False, encoding="utf-8-sig")

raw_df.head()


["Strait of Hormuz"] 2026-01-01 ~ 2026-08-31 | hits=10,000 | pages=10
→ 결과가 많아 날짜를 분할합니다: 2026-01-01~2026-05-02 / 2026-05-03~2026-08-31
  ["Strait of Hormuz"] 2026-01-01 ~ 2026-05-02 | hits=10,000 | pages=10
  → 결과가 많아 날짜를 분할합니다: 2026-01-01~2026-03-02 / 2026-03-03~2026-05-02
    ["Strait of Hormuz"] 2026-01-01 ~ 2026-03-02 | hits=10,000 | pages=10
    → 결과가 많아 날짜를 분할합니다: 2026-01-01~2026-01-31 / 2026-02-01~2026-03-02
      ["Strait of Hormuz"] 2026-01-01 ~ 2026-01-31 | hits=2,714 | pages=3
      ["Strait of Hormuz"] 2026-02-01 ~ 2026-03-02 | hits=10,000 | pages=10
      → 결과가 많아 날짜를 분할합니다: 2026-02-01~2026-02-15 / 2026-02-16~2026-03-02
        ["Strait of Hormuz"] 2026-02-01 ~ 2026-02-15 | hits=4,045 | pages=5
        ["Strait of Hormuz"] 2026-02-16 ~ 2026-03-02 | hits=10,000 | pages=10
        → 결과가 많아 날짜를 분할합니다: 2026-02-16~2026-02-23 / 2026-02-24~2026-03-02
          ["Strait of Hormuz"] 2026-02-16 ~ 2026-02-23 | hits=4,968 | pages=5
          ["Strait of Hormuz"] 2026-02-24 ~ 2026-03-

,title,author,authors,journalists,published_date,published_date_precision,updated_date,updated_date_precision,link,domain_url,...,word_count,is_opinion,twitter_account,all_links,all_domain_links,id,score,duplicate_count,duplicate_articles_group_id,is_content_truncated
0,US warns Iran against unsafe actions during na...,Khaleejtimes,"[Khaleejtimes, system]",[],2026-01-31 00:00:00,date,NaN,NaN,https://article.wn.com/view/2026/01/31/US_warn...,wn.com,...,65,False,@worldnewsdotcom,"[https://travelagents.com/, https://www.clevel...","[travelagents.com, globalweather.com, emission...",0b08bbab067093c463795c247d0aa5d0,13.954778,0.0,bc5fb181e0a94626bb3782ff5a21045f,True
1,"As Iran–US tensions soar, Strait of Hormuz cau...",The Siasat Daily,"[The Siasat Daily, system]",[],2026-01-31 00:00:00,date,NaN,NaN,https://article.wn.com/view/2026/01/31/As_Iran...,wn.com,...,79,False,@worldnewsdotcom,[https://www.publicnow.com/view/478346E02E8C8A...,"[dailymail.co.uk, dubai.com, cities.com, wages...",4bb0a43d1de1f82a121c877b8034a9eb,13.917098,0.0,220b3a2d4f534a10af6b2bf3083a33d6,True
2,What to know about the Strait of Hormuz as Ira...,JON GAMBRELL,[JON GAMBRELL],[Jon Gambrell],2026-01-31 00:00:00,date,2026-01-31 00:00:00,date,https://sitkasentinel.com/stories/what-to-know...,sitkasentinel.com,...,35,False,NaN,[],[],fb5e16653f0da485e647950f55a5141e,13.349553,0.0,a15836cc7c3348a092ff08a9ced55698,True
3,Iran's Foreign Minister Criticizes US Military...,,[],[],2026-01-31 00:00:00,date,NaN,NaN,https://turkiyenewupdates.com/irans-foreign-mi...,turkiyenewupdates.com,...,260,False,NaN,[],[],4a15cf9d2ae915b7ee21ba86c0c33803,13.323380,0.0,c41eedf2a44b48a4a32cb2d64198d724,True
4,Strait of Hormuz becomes centre of Iran's mili...,Hindustan Times,"[system, Hindustan Times]",[],2026-01-31 00:00:00,date,NaN,NaN,https://article.wn.com/view/2026/01/31/Strait_...,wn.com,...,8,False,@worldnewsdotcom,[https://www.independent.co.uk/news/world/midd...,"[students.com, ndtv.com, worldphotos.com, duba...",e2abf87c8b2ec60d9ce90c0d75414b99,12.836854,0.0,5a8a3fe4b7944189a9b51a91324080f5,True


## 5. NewsCatcher 결과를 연구용 기사 목록으로 정리

API 응답 구조가 계정/버전에 따라 일부 달라질 수 있으므로,
아래 코드는 자주 사용되는 필드 후보를 순서대로 찾아 사용합니다.


In [6]:
def first_existing(row, candidates, default=None):
    for c in candidates:
        if c in row.index and pd.notna(row[c]) and str(row[c]).strip():
            return row[c]
    return default

def make_article_index(df):
    records = []

    for _, row in df.iterrows():
        url = first_existing(row, [
            "link", "url", "clean_url", "canonical_url", "parent_url"
        ])

        if not url:
            continue

        url = normalize_url(url)
        if not url:
            continue

        records.append({
            "title_search": first_existing(row, ["title"]),
            "description_search": first_existing(row, ["summary", "excerpt", "content", "description"]),
            "source_search": first_existing(row, ["clean_url", "domain_url", "source_url", "source_name", "source"]),
            "published_search": first_existing(row, ["published_date", "published_at", "pub_date"]),
            "url": url,
            "language_search": first_existing(row, ["language", "lang"], "en"),
        })

    out = pd.DataFrame(records)
    out = out.drop_duplicates(subset=["url"]).reset_index(drop=True)

    out["article_id"] = [
        hashlib.sha1(u.encode("utf-8")).hexdigest()[:16]
        for u in out["url"]
    ]

    return out

article_index = make_article_index(raw_df)

print("URL 중복 제거 후 기사 수:", len(article_index))
article_index.head()


URL 중복 제거 후 기사 수: 427874


,title_search,description_search,source_search,published_search,url,language_search,article_id
0,US warns Iran against unsafe actions during na...,The United States has warned Iran against unsa...,wn.com,2026-01-31 00:00:00,https://article.wn.com/view/2026/01/31/US_warn...,en,6e858ebeef5aa494
1,"As Iran–US tensions soar, Strait of Hormuz cau...","Dubai: The Strait of Hormuz, the narrow mouth ...",wn.com,2026-01-31 00:00:00,https://article.wn.com/view/2026/01/31/As_Iran...,en,884683affb4a1f2b
2,What to know about the Strait of Hormuz as Ira...,Iran plans a military drill in the Strait of H...,sitkasentinel.com,2026-01-31 00:00:00,https://sitkasentinel.com/stories/what-to-know...,en,5e8b21085d75f44d
3,Iran's Foreign Minister Criticizes US Military...,"According to Anadolu Agency, Araghchi expresse...",turkiyenewupdates.com,2026-01-31 00:00:00,https://turkiyenewupdates.com/irans-foreign-mi...,en,6bc553fe61ec7b9e
4,Strait of Hormuz becomes centre of Iran's mili...,Iran recently warned that it will conduct a,wn.com,2026-01-31 00:00:00,https://article.wn.com/view/2026/01/31/Strait_...,en,bf448b12ff11fc71


## 6. Diffbot Article API — 본문/언론사/국가/지역 추출

Diffbot Article API의 핵심 필드는 다음과 같습니다.

- `title`
- `text` = 전체 기사 본문
- `date`
- `author`
- `siteName`
- `publisherCountry`
- `publisherRegion`
- `humanLanguage`
- `resolvedPageUrl`
- `discussion`

기사 본문이 실제로 추출되었는지를 `body_length`와 `extraction_status`로 기록합니다.


In [7]:
def diffbot_article(url, include_discussion=False, timeout_ms=60000):
    """Diffbot Article API로 기사 본문을 추출합니다.

    댓글은 기본적으로 수집하지 않습니다.
    댓글이 필요할 경우 별도의 Discussion API 단계에서 보완합니다.
    """
    params = {
        "token": DIFFBOT_API_KEY,
        "url": url,
        "timeout": timeout_ms,
        "discussion": "true" if include_discussion else "false",
        "paging": "true",
    }

    for attempt in range(5):
        try:
            r = requests.get(
                DIFFBOT_ARTICLE_URL,
                params=params,
                timeout=90,
            )

            if r.status_code == 429:
                if attempt == 4:
                    raise RuntimeError("Diffbot rate limit(429)입니다.")
                wait = min(60, 2 ** attempt * 2)
                print(f"  Diffbot 429 → {wait}초 후 재시도")
                time.sleep(wait)
                continue

            if 500 <= r.status_code < 600:
                if attempt == 4:
                    r.raise_for_status()
                wait = min(60, 2 ** attempt * 2)
                time.sleep(wait)
                continue

            r.raise_for_status()
            data = r.json()

            objects = data.get("objects") or []
            if not objects:
                return None, data

            return objects[0], data

        except requests.RequestException:
            if attempt == 4:
                raise
            wait = min(60, 2 ** attempt * 2)
            time.sleep(wait)

    return None, None


def flatten_article(article_id, source_url, obj):
    if not obj:
        return {
            "article_id": article_id,
            "source_url": source_url,
            "extraction_status": "no_object",
            "body": "",
            "body_length": 0,
        }

    body = obj.get("text") or ""

    return {
        "article_id": article_id,
        "source_url": source_url,
        "resolved_url": obj.get("resolvedPageUrl"),
        "title": obj.get("title"),
        "body": body,
        "body_length": len(body),
        "published_at": obj.get("date"),
        "estimated_date": obj.get("estimatedDate"),
        "author": obj.get("author"),
        "source": obj.get("siteName"),
        "publisher_country": obj.get("publisherCountry"),
        "publisher_region": obj.get("publisherRegion"),
        "language": obj.get("humanLanguage"),
        "num_pages": obj.get("numPages"),
        "extraction_status": (
            "success" if len(body.strip()) >= 100 else "short_or_empty"
        ),
    }


## 7. 본문 수집 실행

중단되더라도 이미 저장된 결과를 다시 요청하지 않도록
`articles_partial.csv`에 주기적으로 저장합니다.

처음에는 **API 테스트 비용을 아끼기 위해 `TEST_LIMIT=20`**으로 실행합니다.
20건의 결과가 정상임을 확인한 뒤 `TEST_LIMIT=None`으로 변경하여 전체 수집합니다.

### 댓글

댓글은 프로젝트의 필수 데이터가 아니므로 **기본적으로 수집하지 않습니다.**
기사 본문 수집이 우선입니다.


In [8]:
TEST_LIMIT = 20  # 테스트 후 전체 수집하려면 None

partial_file = DATA_DIR / "articles_partial.csv"
failed_file = DATA_DIR / "failed_urls_partial.csv"

if partial_file.exists():
    done_df = pd.read_csv(partial_file)
    done_ids = set(done_df["article_id"].astype(str))
else:
    done_df = pd.DataFrame()
    done_ids = set()

targets = article_index[
    ~article_index["article_id"].astype(str).isin(done_ids)
].copy()

if TEST_LIMIT is not None:
    targets = targets.head(TEST_LIMIT)

article_rows = []
failed_rows = []

for _, row in tqdm(
    targets.iterrows(),
    total=len(targets),
    desc="Diffbot Article",
):
    try:
        # 댓글은 여기서 수집하지 않습니다.
        obj, raw = diffbot_article(
            row["url"],
            include_discussion=False,
        )

        article_rows.append(
            flatten_article(row["article_id"], row["url"], obj)
        )

    except Exception as e:
        failed_rows.append({
            "article_id": row["article_id"],
            "source_url": row["url"],
            "error": str(e),
        })

    time.sleep(0.2)

new_articles_df = pd.DataFrame(article_rows)

if len(done_df):
    articles_df = pd.concat(
        [done_df, new_articles_df],
        ignore_index=True,
    )
else:
    articles_df = new_articles_df.copy()

if len(articles_df):
    articles_df = articles_df.drop_duplicates(
        "article_id"
    ).reset_index(drop=True)

articles_df.to_csv(
    partial_file,
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(failed_rows).to_csv(
    failed_file,
    index=False,
    encoding="utf-8-sig",
)

print("현재 Article 결과:", len(articles_df))

if len(articles_df):
    print(
        "본문 100자 이상:",
        (articles_df["body_length"] >= 100).sum(),
    )

print("이번 실행 실패:", len(failed_rows))


Diffbot Article:   5%|▌         | 1/20 [00:05<01:37,  5.11s/it]

  Diffbot 429 → 2초 후 재시도
  Diffbot 429 → 4초 후 재시도
  Diffbot 429 → 8초 후 재시도


Diffbot Article:  10%|█         | 2/20 [00:30<05:08, 17.14s/it]

  Diffbot 429 → 2초 후 재시도
  Diffbot 429 → 4초 후 재시도
  Diffbot 429 → 8초 후 재시도


Diffbot Article:  15%|█▌        | 3/20 [00:52<05:28, 19.34s/it]

...

  Diffbot 429 → 2초 후 재시도
  Diffbot 429 → 4초 후 재시도
  Diffbot 429 → 8초 후 재시도


Diffbot Article:  90%|█████████ | 18/20 [07:03<00:44, 22.33s/it]

  Diffbot 429 → 2초 후 재시도
  Diffbot 429 → 4초 후 재시도


Diffbot Article: 100%|██████████| 20/20 [07:23<00:00, 22.19s/it]

현재 Article 결과: 20
본문 100자 이상: 17
이번 실행 실패: 0


## 8. 댓글은 선택 사항

댓글은 본 연구의 필수 수집 대상이 아닙니다.

기본 실행에서는 Article API에 `discussion=false`를 사용하여
기사 본문 수집에 집중합니다.

댓글 분석이 필요해졌을 때만 아래의 Discussion API를 별도로 실행할 수 있습니다.


In [ ]:
def extract_comments_from_article_object(article_id, obj):
    rows = []

    if not obj:
        return rows

    discussion = obj.get("discussion")
    if not discussion:
        return rows

    posts = discussion.get("posts") or []

    for post in posts:
        text = post.get("text") or ""

        rows.append({
            "article_id": article_id,
            "comment_id": post.get("id"),
            "parent_comment_id": post.get("parentId"),
            "comment_text": text,
            "comment_author": post.get("author"),
            "comment_date": post.get("date"),
            "comment_language": post.get("humanLanguage"),
            "comment_url": post.get("pageUrl"),
        })

    return rows


## 9. 댓글을 확실하게 별도 수집하고 싶을 때

Article API의 `discussion`이 비어 있는 사이트는 Discussion API를 한 번 더 호출할 수 있습니다.
다만 이것은 **모든 기사에 무조건 두 번 요청하지 않도록** 설계하는 것이 좋습니다.

아래 함수는 필요할 때만 Discussion API를 호출합니다.


In [ ]:
def diffbot_discussion(url, timeout_ms=60000):
    params = {
        "token": DIFFBOT_API_KEY,
        "url": url,
        "timeout": timeout_ms,
        "maxPages": "all",
    }

    r = requests.get(DIFFBOT_DISCUSSION_URL, params=params, timeout=90)

    if r.status_code == 429:
        raise RuntimeError("Diffbot Discussion rate limit(429)입니다.")

    r.raise_for_status()
    data = r.json()

    objects = data.get("objects") or []
    return objects[0] if objects else None

def flatten_discussion(article_id, discussion_obj):
    rows = []

    if not discussion_obj:
        return rows

    for post in discussion_obj.get("posts") or []:
        rows.append({
            "article_id": article_id,
            "comment_id": post.get("id"),
            "parent_comment_id": post.get("parentId"),
            "comment_text": post.get("text"),
            "comment_author": post.get("author"),
            "comment_date": post.get("date"),
            "comment_language": post.get("humanLanguage"),
            "comment_url": post.get("pageUrl"),
        })

    return rows


## 10. 현재 Article 결과에서 댓글이 없는 기사만 Discussion API로 보완

이 단계는 API 비용이 추가될 수 있으므로 필요할 때 실행합니다.


In [ ]:
RUN_DISCUSSION_BACKFILL = False  # 필요할 때만 True

comments_rows = []

if RUN_DISCUSSION_BACKFILL:
    for _, row in tqdm(
        articles_df[articles_df["source_url"].notna()].iterrows(),
        total=articles_df["source_url"].notna().sum(),
        desc="Diffbot Discussion",
    ):
        try:
            discussion = diffbot_discussion(row["source_url"])
            comments_rows.extend(
                flatten_discussion(row["article_id"], discussion)
            )
        except Exception as e:
            print("댓글 실패:", row["source_url"], e)

        time.sleep(0.2)

comments_df = pd.DataFrame(comments_rows)

if len(comments_df):
    comments_df = comments_df.drop_duplicates(
        subset=["article_id", "comment_id", "comment_text"]
    )
    comments_df.to_csv(
        COMMENTS_FILE,
        index=False,
        encoding="utf-8-sig",
    )
    print("댓글 수:", len(comments_df))
else:
    print(
        "댓글 수집을 실행하지 않았습니다. "
        "필요한 경우 RUN_DISCUSSION_BACKFILL=True로 변경하세요."
    )


## 11. 수집 결과 검증

최종 수집 결과에서 다음 항목을 확인합니다.

- raw 검색 결과 수
- URL 중복 제거 후 기사 수
- Diffbot 본문 추출 성공 수
- 본문 100자 미만 수
- 영어 기사 수
- 언론사 국가 미상 수
- 언론사명 미상 수
- 수집된 날짜 범위

특히 **2026-01-01부터 2026-08-31까지 실제 날짜가 분포하는지** 확인하는 것이 중요합니다.


In [9]:
# 전체 저장
articles_df.to_csv(
    ARTICLES_FILE,
    index=False,
    encoding="utf-8-sig",
)

if len(articles_df):
    print("===== ARTICLE COLLECTION REPORT =====")
    print("raw 기사 수:", len(raw_df))
    print("URL 중복 제거 후 기사 수:", len(article_index))
    print("Diffbot 기사 수:", len(articles_df))
    print(
        "본문 100자 이상:",
        (articles_df["body_length"] >= 100).sum(),
    )
    print(
        "본문 100자 미만:",
        (articles_df["body_length"] < 100).sum(),
    )
    print(
        "언어 en:",
        (
            articles_df["language"]
            .astype(str)
            .str.lower()
            == "en"
        ).sum(),
    )
    print(
        "국가 미상:",
        articles_df["publisher_country"].isna().sum(),
    )
    print(
        "언론사 미상:",
        articles_df["source"].isna().sum(),
    )

    parsed_dates = pd.to_datetime(
        articles_df["published_at"],
        errors="coerce",
    )

    print("Diffbot 날짜 최소:", parsed_dates.min())
    print("Diffbot 날짜 최대:", parsed_dates.max())

    display(
        articles_df[
            [
                "article_id",
                "title",
                "source",
                "publisher_country",
                "publisher_region",
                "language",
                "published_at",
                "body_length",
                "extraction_status",
            ]
        ].head(20)
    )


===== ARTICLE COLLECTION REPORT =====
raw 기사 수: 440260
URL 중복 제거 후 기사 수: 427874
Diffbot 기사 수: 20
본문 100자 이상: 17
본문 100자 미만: 3
언어 en: 17
국가 미상: 16
언론사 미상: 3
Diffbot 날짜 최소: 2026-01-30 00:00:00
Diffbot 날짜 최대: 2026-09-11 00:00:00


,article_id,title,source,publisher_country,publisher_region,language,published_at,body_length,extraction_status
0,6e858ebeef5aa494,US warns Iran against unsafe actions during na...,wn.com,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",471,success
1,884683affb4a1f2b,"As Iran–US tensions soar, Strait of Hormuz cau...",wn.com,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",486,success
2,5e8b21085d75f44d,What to know about the Strait of Hormuz as Ira...,Daily Sitka Sentinel,NaN,NaN,en,"Wed, 18 Feb 2026 03:35:10 GMT",4451,success
3,6bc553fe61ec7b9e,Iran’s Foreign Minister Criticizes US Military...,Türkiye New Updates,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",2225,success
4,bf448b12ff11fc71,NaN,NaN,NaN,NaN,NaN,NaN,0,no_object
5,722ac8391f9ef3ed,Explosion hits Iran's Bandar Abbas port amid t...,wn.com,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",195,success
6,87e3d3a0d86992da,U.S. warns Iran over unsafe naval actions,anewz.tv,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",2477,success
7,7f6698dde7b072d8,"Blast in Iran port city kills 1, wounds 14 bef...",wn.com,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",395,success
8,2ed76743a1b74307,NaN,NaN,NaN,NaN,NaN,NaN,0,no_object
9,4c96850392df843e,US will 'not tolerate unsafe' actions by Iran'...,Anadolu Ajansi,Turkey,Western Asia,en,"Fri, 30 Jan 2026 22:52:07 GMT",2245,success


## 12. Excel 파일 생성

분석용으로는 CSV가 더 안정적이고,
확인/제출용으로는 Excel을 같이 만들어 둡니다.


In [10]:
excel_path = DATA_DIR / "hormuz_2026_articles_comments.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    articles_df.to_excel(writer, sheet_name="articles", index=False)

    if COMMENTS_FILE.exists():
        comments_for_excel = pd.read_csv(COMMENTS_FILE)
        comments_for_excel.to_excel(writer, sheet_name="comments", index=False)

print("저장 완료:", excel_path)


저장 완료: hormuz_2026_data\hormuz_2026_articles_comments.xlsx


# 13. 기존 텍스트마이닝 코드와 연결

여기부터는 기존 노트북의 NLTK 전처리 코드를 연결하면 됩니다.

중요한 점은 `body`를 분석 대상으로 사용하고,
`title`은 별도 분석 컬럼으로 보존하는 것입니다.

또한 `hormuz`, `strait`은 연구 질문에 따라 불용어로 제거할 수 있지만,
`iran`, `iranian`, `israel`, `israeli`, `us`, `america` 등을 무조건 제거하면
국가/행위자 프레임 분석에 필요한 정보가 사라질 수 있으므로 신중하게 결정합니다.


In [ ]:
# 텍스트마이닝 시작용 최소 코드

analysis_df = articles_df.copy()

# 실제 본문이 충분한 기사만 분석
analysis_df = analysis_df[
    (analysis_df["language"].astype(str).str.lower() == "en") &
    (analysis_df["body_length"] >= 100)
].copy()

analysis_df["text_for_mining"] = (
    analysis_df["title"].fillna("") + " " +
    analysis_df["body"].fillna("")
)

print("텍스트마이닝 대상 기사:", len(analysis_df))
analysis_df[[
    "title", "source", "publisher_country",
    "publisher_region", "published_at", "body_length"
]].head()


## 14. 중단 후 재시작

`articles_partial.csv`가 남아 있으면 이미 처리된 `article_id`는 다시 요청하지 않습니다.

따라서 전체 수집 중 Jupyter/VS Code가 종료되어도
처음부터 다시 시작할 필요가 없습니다.

### 권장 실행 순서

1. `TEST_LIMIT = 20`
2. 20건의 Article 본문 추출 결과 확인
3. 문제가 없으면 `TEST_LIMIT = None`
4. NewsCatcher 전체 기간 수집
5. 날짜 분할 로그에서 2026-01-01 ~ 2026-08-31이 빠짐없이 처리되는지 확인
6. Diffbot Article 전체 수집
7. `RUN_DISCUSSION_BACKFILL = False` 유지
8. 댓글 분석이 필요할 때만 별도로 Discussion API 실행
9. 최종 CSV/Excel 생성


In [11]:
import os
import requests

DIFFBOT_API_KEY = os.getenv("DIFFBOT_API_KEY")

url = "https://api.diffbot.com/v4/account"

response = requests.get(
    url,
    params={
        "token": DIFFBOT_API_KEY,
        "days": 31
},
    timeout=30
)

print("HTTP 상태 코드:", response.status_code)

if response.ok:
    data = response.json()

    print("플랜:", data.get("plan"))
    print("상태:", data.get("status"))
    print("월간 포함 크레딧:", data.get("planCredits"))

    usage = data.get("usage", [])

    if usage:
        total_credits = sum(
            item.get("credits", 0)
            for item in usage
        )

        total_extractions = sum(
            item.get("extractions", 0)
            for item in usage
        )

        print("최근 31일 사용 크레딧:", total_credits)
        print("최근 31일 Extract 호출:", total_extractions)

        print("\n최근 사용량:")
        for item in usage[:10]:
            print(
                item.get("date"),
                "credits =", item.get("credits", 0),
                "extractions =", item.get("extractions", 0)
            )
else:
    print("Diffbot API 오류")
    print(response.text)

HTTP 상태 코드: 200
플랜: kgfree
상태: active
월간 포함 크레딧: 10000
최근 31일 사용 크레딧: 20
최근 31일 Extract 호출: 20

최근 사용량:
2026-09-11 credits = 20 extractions = 20
2026-09-10 credits = 0 extractions = 0
2026-09-09 credits = 0 extractions = 0
2026-09-08 credits = 0 extractions = 0
2026-09-07 credits = 0 extractions = 0
2026-09-06 credits = 0 extractions = 0
2026-09-05 credits = 0 extractions = 0
2026-09-04 credits = 0 extractions = 0
2026-09-03 credits = 0 extractions = 0
2026-09-02 credits = 0 extractions = 0
